# EDA - Funil SaaS Conta Azul

Este notebook demonstra a investigacao analitica do case de Business Analytics. Ele complementa o dashboard Streamlit, mostrando o raciocinio por tras dos principais achados.

Objetivos:

- Ler o CSV original com Pandas.
- Fazer profiling da base.
- Validar qualidade e consistencia dos dados.
- Calcular o funil geral.
- Analisar conversao por canal, dispositivo, pais e mes.
- Analisar NPS de compradores elegiveis e respostas nao elegiveis.
- Gerar graficos com Plotly.
- Consolidar conclusoes antes do dashboard.


In [1]:
from pathlib import Path
import sys

import duckdb
import pandas as pd
import plotly.express as px
import plotly.io as pio

ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == 'notebooks':
    ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


DATA_FILE = ROOT_DIR / 'saas_funnel_case_10k_refresh_(4)_(2).csv'

# Paleta inspirada na identidade visual da Conta Azul.
CONTA_AZUL_COLORS = [
    '#1B69C8',  # azul principal
    '#14A7E0',  # azul claro
    '#00B8D9',  # ciano
    '#49C96D',  # verde de acao/positivo
    '#FDB913',  # amarelo de destaque
    '#0B2F6B',  # azul escuro
]
px.defaults.color_discrete_sequence = CONTA_AZUL_COLORS
px.defaults.template = 'plotly_white'
pio.renderers.default = 'vscode'
print(f"Arquivo CSV encontrado em: {DATA_FILE}")
print(f"Arquivo existe? {DATA_FILE.exists()}")

Arquivo CSV encontrado em: C:\Projetos\desafio-conta-azul\saas_funnel_case_10k_refresh_(4)_(2).csv
Arquivo existe? True


## 1. Leitura do CSV com Pandas

In [2]:
raw = pd.read_csv(DATA_FILE)
raw.head()

,user_id,dt_visit,channel,device,country,signup,purchase,plan,days_to_signup,days_to_purchase,respondeu_nps,nps_score
0,1,2025-07-21,organic,mobile,BR,0,0,NaN,NaN,NaN,0,NaN
1,2,2025-07-28,referral,desktop,BR,0,0,NaN,NaN,NaN,0,NaN
2,3,2025-07-05,paid,desktop,BR,0,0,NaN,NaN,NaN,0,NaN
3,4,2025-07-27,paid,mobile,BR,0,0,NaN,NaN,NaN,0,NaN
4,5,2025-10-25,organic,mobile,BR,0,0,NaN,NaN,NaN,0,NaN


In [3]:
pd.DataFrame({
    "metrica": ["linhas", "colunas"],
    "valor": [raw.shape[0], raw.shape[1]]
})

,metrica,valor
0,linhas,10000
1,colunas,12


## 2. Tratamento inicial e campos derivados

A base tem grao de uma linha por usuario visitante. Criamos campos derivados para facilitar a leitura de funil, tempo e NPS.

In [4]:
df = raw.copy()
df['dt_visit'] = pd.to_datetime(df['dt_visit'], errors='coerce')
df['days_to_signup'] = df['days_to_signup'].astype('Int64')
df['days_to_purchase'] = df['days_to_purchase'].astype('Int64')
df['nps_score'] = df['nps_score'].astype('Float64')

df['visit_month'] = df['dt_visit'].dt.to_period('M').astype(str)
df['signup_date'] = df['dt_visit'] + pd.to_timedelta(df['days_to_signup'], unit='D')
df['purchase_date'] = df['signup_date'] + pd.to_timedelta(df['days_to_purchase'], unit='D')

df['funnel_stage'] = 'Visit only'
df.loc[(df['signup'] == 1) & (df['purchase'] == 0), 'funnel_stage'] = 'Signup only'
df.loc[df['purchase'] == 1, 'funnel_stage'] = 'Purchased'

df['nps_class'] = 'No response'
df.loc[df['nps_score'].notna() & (df['nps_score'] < 7), 'nps_class'] = 'Detractor'
df.loc[df['nps_score'].notna() & (df['nps_score'] >= 7) & (df['nps_score'] < 9), 'nps_class'] = 'Passive'
df.loc[df['nps_score'].notna() & (df['nps_score'] >= 9), 'nps_class'] = 'Promoter'

df.head()

,user_id,dt_visit,channel,device,country,signup,purchase,plan,days_to_signup,days_to_purchase,respondeu_nps,nps_score,visit_month,signup_date,purchase_date,funnel_stage,nps_class
0,1,2025-07-21,organic,mobile,BR,0,0,NaN,<NA>,<NA>,0,<NA>,2025-07,NaT,NaT,Visit only,No response
1,2,2025-07-28,referral,desktop,BR,0,0,NaN,<NA>,<NA>,0,<NA>,2025-07,NaT,NaT,Visit only,No response
2,3,2025-07-05,paid,desktop,BR,0,0,NaN,<NA>,<NA>,0,<NA>,2025-07,NaT,NaT,Visit only,No response
3,4,2025-07-27,paid,mobile,BR,0,0,NaN,<NA>,<NA>,0,<NA>,2025-07,NaT,NaT,Visit only,No response
4,5,2025-10-25,organic,mobile,BR,0,0,NaN,<NA>,<NA>,0,<NA>,2025-10,NaT,NaT,Visit only,No response


## 3. Profiling da base

In [5]:
profile = pd.DataFrame({
    'metrica': [
        'linhas', 'colunas', 'usuarios_unicos', 'usuarios_duplicados',
        'data_minima_visita', 'data_maxima_visita', 'canais', 'dispositivos', 'paises', 'planos'
    ],
    'valor': [
        len(df), df.shape[1], df['user_id'].nunique(), df['user_id'].duplicated().sum(),
        df['dt_visit'].min().date(), df['dt_visit'].max().date(),
        df['channel'].nunique(), df['device'].nunique(), df['country'].nunique(), df['plan'].nunique(dropna=True)
    ]
})
profile

,metrica,valor
0,linhas,10000
1,colunas,17
2,usuarios_unicos,10000
3,usuarios_duplicados,0
4,data_minima_visita,2025-06-01
5,data_maxima_visita,2025-10-30
6,canais,5
7,dispositivos,2
8,paises,5
9,planos,3


In [6]:
df[['channel', 'device', 'country', 'plan']].describe(include='all')

,channel,device,country,plan
count,10000,10000,10000,919
unique,5,2,5,3
top,organic,mobile,BR,BASIC
freq,4271,6242,6986,526


## 4. Validacoes de qualidade

As validacoes abaixo garantem que as metricas de funil nao estao contaminadas por inconsistencias basicas.

In [7]:
checks = pd.DataFrame([
    ('purchase_sem_signup', ((df['purchase'] == 1) & (df['signup'] == 0)).sum()),
    ('plan_sem_purchase', (df['plan'].notna() & (df['purchase'] == 0)).sum()),
    ('purchase_sem_plan', ((df['purchase'] == 1) & df['plan'].isna()).sum()),
    ('days_signup_sem_signup', (df['days_to_signup'].notna() & (df['signup'] == 0)).sum()),
    ('signup_sem_days_signup', ((df['signup'] == 1) & df['days_to_signup'].isna()).sum()),
    ('days_purchase_sem_purchase', (df['days_to_purchase'].notna() & (df['purchase'] == 0)).sum()),
    ('purchase_sem_days_purchase', ((df['purchase'] == 1) & df['days_to_purchase'].isna()).sum()),
    ('nps_score_sem_resposta', (df['nps_score'].notna() & (df['respondeu_nps'] == 0)).sum()),
    ('resposta_sem_nps_score', ((df['respondeu_nps'] == 1) & df['nps_score'].isna()).sum()),
    ('nps_fora_intervalo', ((df['nps_score'] < 0) | (df['nps_score'] > 10)).sum()),
    ('respostas_nps_nao_compradores', ((df['respondeu_nps'] == 1) & (df['purchase'] == 0)).sum()),
], columns=['regra', 'resultado'])

checks

,regra,resultado
0,purchase_sem_signup,0
1,plan_sem_purchase,0
2,purchase_sem_plan,0
3,days_signup_sem_signup,0
4,signup_sem_days_signup,0
5,days_purchase_sem_purchase,0
6,purchase_sem_days_purchase,0
7,nps_score_sem_resposta,0
8,resposta_sem_nps_score,0
9,nps_fora_intervalo,0


## 5. DuckDB: SQL local e views analiticas

A partir daqui usamos DuckDB para calcular as principais metricas em SQL, mantendo a analise reproduzivel.

In [8]:
con = duckdb.connect(database=':memory:')
con.register('stg_funnel_users', df)

create_views_sql = (ROOT_DIR / 'sql' / '01_create_views.sql').read_text(encoding='utf-8')
con.execute(create_views_sql)

con.execute("select table_name from information_schema.tables order by table_name").df()

,table_name
0,stg_funnel_users
1,vw_channel_device
2,vw_funnel_by_channel
3,vw_funnel_by_country
4,vw_funnel_by_device
5,vw_funnel_by_month
6,vw_funnel_overall
7,vw_nps_by_channel
8,vw_nps_summary
9,vw_plan_mix


## 6. Funil geral

In [9]:
overall = con.execute('select * from vw_funnel_overall').df()
overall

,visits,signups,purchases,nps_responses,visit_to_signup_rate,visit_to_purchase_rate,signup_to_purchase_rate,avg_nps_score
0,10000,2983.0,919.0,1206.0,0.2983,0.0919,0.308079,8.108789


In [10]:
funnel_steps = pd.DataFrame({
    'etapa': ['Visitas', 'Signups', 'Compras'],
    'usuarios': [int(overall.loc[0, 'visits']), int(overall.loc[0, 'signups']), int(overall.loc[0, 'purchases'])]
})

fig = px.funnel(funnel_steps, x='usuarios', y='etapa', title='Funil geral - visitas, signups e compras',
    color_discrete_sequence=CONTA_AZUL_COLORS
)
fig.show()

Leitura: o funil apresenta 10.000 visitas, 2.983 signups e 919 compras. O maior gargalo absoluto esta antes do signup, seguido pela perda pos-signup.

## 7. Analise por canal

In [11]:
channel = con.execute('select * from vw_funnel_by_channel order by visit_to_purchase_rate desc').df()
channel

,channel,visits,signups,purchases,visit_to_signup_rate,visit_to_purchase_rate,signup_to_purchase_rate,avg_nps_score
0,referral,982,409.0,179.0,0.416497,0.182281,0.437653,8.375897
1,email,501,188.0,71.0,0.375250,0.141717,0.377660,8.060000
2,organic,4271,1339.0,459.0,0.313510,0.107469,0.342793,8.166841
3,paid,3247,828.0,169.0,0.255005,0.052048,0.204106,7.845018
4,social,999,219.0,41.0,0.219219,0.041041,0.187215,7.985714


In [12]:
fig = px.bar(
    channel.sort_values('visit_to_purchase_rate'),
    x='visit_to_purchase_rate',
    y='channel',
    orientation='h',
    text='purchases',
    title='Visit to purchase por canal',
    color_discrete_sequence=CONTA_AZUL_COLORS
)
fig.update_xaxes(tickformat='.0%')
fig.show()

Leitura: `referral` e `email` sao os canais mais eficientes. `organic` tem forte volume absoluto. `paid` e `social` precisam de otimizacao antes de escala.

## 8. Analise por dispositivo

In [13]:
device = con.execute('select * from vw_funnel_by_device order by visit_to_purchase_rate desc').df()
device

,device,visits,signups,purchases,visit_to_signup_rate,visit_to_purchase_rate,signup_to_purchase_rate,avg_nps_score
0,desktop,3758,1359.0,493.0,0.361629,0.131187,0.362767,8.177778
1,mobile,6242,1624.0,426.0,0.260173,0.068247,0.262315,8.045714


In [14]:
fig = px.bar(
    device,
    x='device',
    y=['visit_to_signup_rate', 'visit_to_purchase_rate', 'signup_to_purchase_rate'],
    barmode='group',
    title='Taxas de conversao por dispositivo',
    color_discrete_sequence=CONTA_AZUL_COLORS
)
fig.update_yaxes(tickformat='.0%')
fig.show()

Leitura: desktop converte melhor que mobile. Como mobile concentra mais visitas, melhorar esse fluxo pode gerar impacto relevante.

## 9. Analise por pais e por mes

In [15]:
country = con.execute('select * from vw_funnel_by_country order by visits desc').df()
country

,country,visits,signups,purchases,visit_to_signup_rate,visit_to_purchase_rate,signup_to_purchase_rate,avg_nps_score
0,BR,6986,2084.0,644.0,0.298311,0.092184,0.309021,8.121287
1,MX,1009,308.0,95.0,0.305253,0.094153,0.308442,7.994068
2,AR,807,231.0,74.0,0.286245,0.091698,0.320346,8.012500
3,CL,708,220.0,58.0,0.310734,0.081921,0.263636,8.154118
4,US,490,140.0,48.0,0.285714,0.097959,0.342857,8.233333


In [16]:
monthly = con.execute('select * from vw_funnel_by_month order by visit_month').df()
monthly

,visit_month,visits,signups,purchases,visit_to_signup_rate,visit_to_purchase_rate,signup_to_purchase_rate,avg_nps_score
0,2025-06,1872,581.0,164.0,0.310363,0.087607,0.282272,7.939914
1,2025-07,2053,593.0,165.0,0.288846,0.080370,0.278246,8.026400
2,2025-08,2121,615.0,201.0,0.289958,0.094767,0.326829,8.073993
3,2025-09,1955,607.0,211.0,0.310486,0.107928,0.347611,8.215299
4,2025-10,1999,587.0,178.0,0.293647,0.089045,0.303237,8.333516


In [17]:
fig = px.line(
    monthly,
    x='visit_month',
    y=['visit_to_signup_rate', 'visit_to_purchase_rate', 'signup_to_purchase_rate'],
    markers=True,
    title='Evolucao mensal das taxas de conversao',
    color_discrete_sequence=CONTA_AZUL_COLORS
)
fig.update_yaxes(tickformat='.0%')
fig.show()

Leitura: a evolucao mensal mostra que setembro apresenta a melhor taxa de compra sobre visitas e a melhor conversao pos-signup. A comparacao por pais foi feita na tabela anterior; aqui o foco e a variacao ao longo dos meses.

## 10. Analise de NPS

In [ ]:
nps_summary = con.execute("""
select * from vw_nps_summary
""").df()

nps_eligibility = con.execute("""
select * from vw_nps_eligibility
""").df()

display(nps_summary)
display(nps_eligibility)


In [ ]:
fig = px.bar(
    nps_summary,
    x='segmento',
    y='nps',
    text='nps',
    title='NPS de compradores elegiveis',
    color_discrete_sequence=CONTA_AZUL_COLORS
)
fig.update_traces(texttemplate='%{text:.1f}')
fig.show()


In [20]:
nps_channel = con.execute('select * from vw_nps_by_channel order by nps desc').df()
nps_channel

,channel,respostas,nps_medio,promotores,detratores,nps
0,referral,195,8.375897,62.0,21.0,21.025641
1,organic,573,8.166841,171.0,102.0,12.041885
2,email,90,8.060000,26.0,16.0,11.111111
3,social,77,7.985714,19.0,20.0,-1.298701
4,paid,271,7.845018,59.0,68.0,-3.321033


Leitura: NPS foi tratado como pesquisa pos-compra. Por isso, o indicador final considera apenas compradores elegiveis. Respostas associadas a usuarios sem compra aparecem como ponto de investigacao de regra de negocio, janela de atribuicao ou tracking, e nao como NPS valido de nao compradores.

## 11. Principais conclusoes antes do dashboard

1. O funil tem dois gargalos relevantes: visita para signup e signup para compra.
2. `referral` e o canal de maior qualidade e deveria ser priorizado em experimentos de crescimento.
3. `paid` tem volume, mas baixa eficiencia; a recomendacao e otimizar antes de escalar investimento.
4. Mobile tem alto volume e baixa conversao relativa, indicando oportunidade de UX, formulario, onboarding ou checkout.
5. `organic` combina escala e eficiencia, sendo um canal importante para manter e evoluir.
6. O NPS deve ser calculado apenas para compradores elegiveis quando a pesquisa e pos-compra.
7. Respostas NPS associadas a usuarios sem compra devem ser investigadas como regra de negocio ou tracking, nao como NPS de nao compradores.
8. As recomendacoes de negocio devem focar em referral, paid, mobile, organic, email/lifecycle e investigacao qualitativa dos usuarios que abandonam a jornada.
